<a href="https://colab.research.google.com/github/ergul13/mr_akgul/blob/main/veri_seti_indirme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get update
!apt-get install -y google-chrome-stable
!pip install selenium webdriver-manager

OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,212 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,785 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:14 https:

In [ ]:
import os
import shutil
import requests
import re
from concurrent.futures import ThreadPoolExecutor
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

ana_klasor = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri'
hedef_klasor_drive = os.path.join(ana_klasor, 'Saglik_Bakanligi_Verileri')
os.makedirs(hedef_klasor_drive, exist_ok=True)

gecici_klasor = '/content/saglik_gecici'
os.makedirs(gecici_klasor, exist_ok=True)

chrome_options = Options()
chrome_options.add_argument('--headless=new')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

driver.get("https://acikveri.saglik.gov.tr/Home/DataSetDetail/3")
driver.implicitly_wait(10)

butonlar = driver.find_elements(By.XPATH, "//a[contains(@href, 'Download')]")
indirme_linkleri = [buton.get_attribute('href') for buton in butonlar if buton.get_attribute('href')]

selenium_cookies = driver.get_cookies()
user_agent = driver.execute_script("return navigator.userAgent;")
driver.quit()

session = requests.Session()
session.headers.update({"User-Agent": user_agent})
for cookie in selenium_cookies:
    session.cookies.set(cookie['name'], cookie['value'])

def dosya_indir(link):
    try:
        response = session.get(link, stream=True, verify=False, timeout=(15, 60))
        response.raise_for_status()

        dosya_adi = link.split('/')[-1]
        cd = response.headers.get('content-disposition', '')

        if 'filename*=' in cd:
            dosya_adi = cd.split("''")[-1]
        elif 'filename=' in cd:
            match = re.findall('filename="?([^";]+)"?', cd)
            if match:
                dosya_adi = match[0]
        else:
            dosya_adi += ".csv"

        kayit_yolu_gecici = os.path.join(gecici_klasor, dosya_adi)
        kayit_yolu_drive = os.path.join(hedef_klasor_drive, dosya_adi)

        if os.path.exists(kayit_yolu_drive) and os.path.getsize(kayit_yolu_drive) > 1024:
            response.close()
            return f"Atlandı: {dosya_adi}"

        indirilen_boyut = 0
        yazdirma_esigi = 50 * 1024 * 1024
        son_yazdirma = 0

        with open(kayit_yolu_gecici, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    indirilen_boyut += len(chunk)
                    if indirilen_boyut - son_yazdirma >= yazdirma_esigi:
                        print(f"[{dosya_adi}] -> {indirilen_boyut / (1024*1024):.1f} MB iniyor...")
                        son_yazdirma = indirilen_boyut

        shutil.move(kayit_yolu_gecici, kayit_yolu_drive)
        return f"Tamamlandı: {dosya_adi}"
    except Exception as e:
        return f"Hata: {link} - {e}"

print(f"Toplam {len(indirme_linkleri)} dosya bulundu. 8'li paralel indirme başlatılıyor.")

with ThreadPoolExecutor(max_workers=1) as executor:
    sonuclar = executor.map(dosya_indir, indirme_linkleri)
    for sonuc in sonuclar:
        print(sonuc)

Toplam 171 dosya bulundu. 8'li paralel indirme başlatılıyor.
Atlandı: Bilgi.zip
Atlandı: MG_EGITIM_1.zip.001
Atlandı: MG_EGITIM_1.zip.002
Atlandı: MG_EGITIM_1.zip.003
Atlandı: MG_EGITIM_1.zip.004
Atlandı: MG_EGITIM_1.zip.005
Atlandı: MG_EGITIM_1.zip.006
Atlandı: MG_EGITIM_1.zip.007
Atlandı: MG_EGITIM_1.zip.008
Atlandı: MG_EGITIM_1.zip.009
Atlandı: MG_EGITIM_1.zip.010
Atlandı: MG_EGITIM_1.zip.011
Atlandı: MG_EGITIM_1.zip.012
Atlandı: MG_EGITIM_1.zip.014
Atlandı: MG_EGITIM_1.zip.015
Atlandı: MG_EGITIM_1.zip.016
Atlandı: MG_EGITIM_1.zip.017
[MG_YARISMA.zip.001] -> 101.1 MB iniyor...
Atlandı: MG_EGITIM_1.zip.018
Atlandı: MG_EGITIM_1.zip.019
Atlandı: MG_EGITIM_1.zip.020
Atlandı: MG_EGITIM_1.zip.021
Atlandı: MG_EGITIM_1.zip.022
Atlandı: MG_EGITIM_1.zip.023
Atlandı: MG_EGITIM_1.zip.024
Atlandı: MG_EGITIM_1.zip.025
Atlandı: MG_EGITIM_1.zip.026
Atlandı: MG_EGITIM_1.zip.027
Atlandı: MG_EGITIM_1.zip.028
Atlandı: MG_EGITIM_1.zip.029
Atlandı: MG_EGITIM_1.zip.031
Atlandı: MG_EGITIM_1.zip.032
Atlandı

KeyboardInterrupt: 

In [ ]:
import os
import shutil
import requests
import re
from concurrent.futures import ThreadPoolExecutor
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

ana_klasor = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri'
hedef_klasor_drive = os.path.join(ana_klasor, 'Saglik_Bakanligi_Verileri')
os.makedirs(hedef_klasor_drive, exist_ok=True)

gecici_klasor = '/content/saglik_gecici'
os.makedirs(gecici_klasor, exist_ok=True)

chrome_options = Options()
chrome_options.add_argument('--headless=new')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

driver.get("https://acikveri.saglik.gov.tr/Home/DataSetDetail/3")
driver.implicitly_wait(10)

butonlar = driver.find_elements(By.XPATH, "//a[contains(@href, 'Download')]")
indirme_linkleri = [buton.get_attribute('href') for buton in butonlar if buton.get_attribute('href')]

selenium_cookies = driver.get_cookies()
user_agent = driver.execute_script("return navigator.userAgent;")
driver.quit()

session = requests.Session()
session.headers.update({"User-Agent": user_agent})
for cookie in selenium_cookies:
    session.cookies.set(cookie['name'], cookie['value'])

aranan_dosya = "MG_EGITIM_2.zip.064"

def dosya_indir(link):
    try:
        response = session.get(link, stream=True, verify=False, timeout=(15, 60))
        response.raise_for_status()

        dosya_adi = link.split('/')[-1]
        cd = response.headers.get('content-disposition', '')

        if 'filename*=' in cd:
            dosya_adi = cd.split("''")[-1]
        elif 'filename=' in cd:
            match = re.findall('filename="?([^";]+)"?', cd)
            if match:
                dosya_adi = match[0]
        else:
            dosya_adi += ".csv"

        if dosya_adi != aranan_dosya:
            response.close()
            return None

        kayit_yolu_gecici = os.path.join(gecici_klasor, dosya_adi)
        kayit_yolu_drive = os.path.join(hedef_klasor_drive, dosya_adi)

        if os.path.exists(kayit_yolu_drive) and os.path.getsize(kayit_yolu_drive) > 1024:
            response.close()
            return f"Zaten mevcut: {dosya_adi}"

        indirilen_boyut = 0
        yazdirma_esigi = 50 * 1024 * 1024
        son_yazdirma = 0

        with open(kayit_yolu_gecici, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    indirilen_boyut += len(chunk)
                    if indirilen_boyut - son_yazdirma >= yazdirma_esigi:
                        print(f"[{dosya_adi}] -> {indirilen_boyut / (1024*1024):.1f} MB iniyor...")
                        son_yazdirma = indirilen_boyut

        shutil.move(kayit_yolu_gecici, kayit_yolu_drive)
        return f"Tamamlandı: {dosya_adi}"
    except Exception as e:
        return f"Hata: {link} - {e}"

print(f"Sadece {aranan_dosya} aranıyor...")

with ThreadPoolExecutor(max_workers=8) as executor:
    sonuclar = executor.map(dosya_indir, indirme_linkleri)
    for sonuc in sonuclar:
        if sonuc:
            print(sonuc)

In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [6]:
import os
import subprocess
import sys
import shutil

ana_klasor = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri'
indirilenler_klasoru = os.path.join(ana_klasor, 'Saglik_Bakanligi_Verileri')
hedef_klasor = os.path.join(ana_klasor, 'Cikarilan_Veriler')

os.makedirs(hedef_klasor, exist_ok=True)

dosyalar = os.listdir(indirilenler_klasoru)
dosyalar.sort()

arsivler = [f for f in dosyalar if f.endswith('.zip') or f.endswith('.zip.001')]
csvler = [f for f in dosyalar if f.endswith('.csv')]

print(f"Toplam {len(arsivler)} arşiv ve {len(csvler)} CSV dosyası bulundu.\n")

if csvler:
    print("--- CSV Dosyaları Aktarılıyor ---")
    for csv in csvler:
        kaynak_csv = os.path.join(indirilenler_klasoru, csv)
        hedef_csv = os.path.join(hedef_klasor, csv)
        if not os.path.exists(hedef_csv):
            shutil.copy2(kaynak_csv, hedef_csv)
            print(f"[KOPYALANDI] {csv}")

print("\n--- Arşiv Çıkarma İşlemi Başlıyor ---")
for dosya in arsivler:
    klasor_adi = dosya.replace('.zip.001', '').replace('.zip', '')
    hedef_alt_klasor = os.path.join(hedef_klasor, klasor_adi)
    marker_dosya = os.path.join(hedef_alt_klasor, 'TAMAMLANDI.txt')

    # Eğer TAMAMLANDI.txt varsa, bu arşiv sorunsuz çıkarılmıştır, tamamen atla.
    if os.path.exists(marker_dosya):
        print(f"\n[ATLANDI] {klasor_adi} zaten tam olarak çıkarılmış.")
        continue

    os.makedirs(hedef_alt_klasor, exist_ok=True)

    tam_yol = os.path.join(indirilenler_klasoru, dosya)
    print(f"\n{'-'*60}\nİşleniyor: {dosya} -> {klasor_adi}\n{'-'*60}")

    # -aos parametresi: Klasörde zaten var olan dosyaları atla (yarım kalanlar için hızlandırır)
    komut = ['7z', 'x', tam_yol, f'-o{hedef_alt_klasor}', '-aos', '-bsp1']

    process = subprocess.Popen(komut, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)

    for line in process.stdout:
        line = line.strip()
        if line:
            if '%' in line or 'Extracting' in line:
                sys.stdout.write(f"\r{line[:80]:<80}")
                sys.stdout.flush()

    process.wait()

    if process.returncode == 0:
        # İşlem sıfır hata ile biterse işaret dosyasını oluştur
        with open(marker_dosya, 'w') as f:
            f.write('Bu klasor eksiksiz cikarildi.')
        print(f"\n[BAŞARILI] {dosya} eksiksiz çıkarıldı.")
    else:
        print(f"\n[HATA] {dosya} çıkarılırken bağlantı koptu veya hata oluştu.")

print("\nTüm işlemler bitti.")

Toplam 5 arşiv ve 0 CSV dosyası bulundu.


--- Arşiv Çıkarma İşlemi Başlıyor ---

[ATLANDI] Bilgi zaten tam olarak çıkarılmış.

[ATLANDI] MG_EGITIM_1 zaten tam olarak çıkarılmış.

------------------------------------------------------------
İşleniyor: MG_EGITIM_1.zip.001; filename*=UTF-8''MG_EGITIM_1.zip.001 -> MG_EGITIM_1; filename*=UTF-8''MG_EGITIM_1
------------------------------------------------------------
10%     14% 15        30% 30        41% 39
[HATA] MG_EGITIM_1.zip.001; filename*=UTF-8''MG_EGITIM_1.zip.001 çıkarılırken bağlantı koptu veya hata oluştu.

------------------------------------------------------------
İşleniyor: MG_EGITIM_2.zip.001 -> MG_EGITIM_2
------------------------------------------------------------
0%      0% 2 - MG_EGITIM_2/822670054/LCC.dcm
[BAŞARILI] MG_EGITIM_2.zip.001 eksiksiz çıkarıldı.

------------------------------------------------------------
İşleniyor: MG_YARISMA.zip.001

KeyboardInterrupt: 

In [ ]:
import os
import pydicom
import numpy as np
from pathlib import Path
from pydicom.pixel_data_handlers.util import apply_voi_lut
import cv2
from concurrent.futures import ProcessPoolExecutor

def process_single_dicom(args):
    dicom_path, output_directory, input_directory = args
    try:
        dicom = pydicom.dcmread(dicom_path)

        try:
            data = apply_voi_lut(dicom.pixel_array, dicom)
        except:
            data = dicom.pixel_array

        if hasattr(dicom, 'PhotometricInterpretation') and dicom.PhotometricInterpretation == "MONOCHROME1":
            data = np.amax(data) - data

        data = data.astype(np.float32)
        data_min = np.min(data)
        data_max = np.max(data)

        if data_max > data_min:
            data = (data - data_min) / (data_max - data_min)
        else:
            data = np.zeros_like(data)

        data = (data * 255).astype(np.uint8)

        relative_path = dicom_path.relative_to(input_directory)
        output_filename = str(relative_path.with_suffix('.png')).replace(os.sep, '_')
        output_path = os.path.join(output_directory, output_filename)

        cv2.imwrite(output_path, data)
        return True
    except:
        return False

def convert_dicom_to_png_parallel(input_dir, output_dir, workers=4):
    os.makedirs(output_dir, exist_ok=True)

    dicom_files = list(Path(input_dir).rglob('*.dcm'))
    toplam_dosya = len(dicom_files)

    print(f"Toplam {toplam_dosya} DICOM dosyası bulundu. Dönüştürme başlıyor...\n")

    args_list = [(path, output_dir, input_dir) for path in dicom_files]

    basarili = 0
    hatali = 0

    with ProcessPoolExecutor(max_workers=workers) as executor:
        for i, sonuc in enumerate(executor.map(process_single_dicom, args_list)):
            if sonuc:
                basarili += 1
            else:
                hatali += 1

            if (i + 1) % 100 == 0 or (i + 1) == toplam_dosya:
                print(f"İlerleme: {i + 1}/{toplam_dosya} (Başarılı: {basarili}, Hatalı: {hatali})")

    print(f"\nİşlem bitti. {basarili} dosya PNG'ye dönüştürüldü.")

ana_klasor = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri'
dicom_klasoru = os.path.join(ana_klasor, 'Cikarilan_Veriler')
png_klasoru = os.path.join(ana_klasor, 'PNG_Görüntüler')

convert_dicom_to_png_parallel(dicom_klasoru, png_klasoru, workers=4)